In [1]:
# %matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
from matplotlib.gridspec import GridSpec,GridSpecFromSubplotSpec
from matplotlib import animation, rc
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d import axes3d

from sklearn.cluster import OPTICS, cluster_optics_dbscan

from astropy.table import Table
import astropy.units as u
import astropy.constants as const
import corner

from tqdm.notebook import trange, tqdm
import re
import warnings
import os
import importlib
import VarStar_Vi_plot_functions as vi
from subprocess import *

In [2]:
TDSS_Table = Table.read('/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Statistical_Testing/LC_stat_calculations/Control_StatsCalc/TDSS_VarStar_FINAL_Var_ALL_PROP_STATS_03-15-2022.fits')
TDSS_Table.add_column(np.arange(len(TDSS_Table)), name='prop_id', index=0)

selection_index = (TDSS_Table['ZTF_g_logProb']>=-5).data & (TDSS_Table['ZTF_r_logProb']>=-5).data & (TDSS_Table['CSS_logProb']>=-5).data
print(selection_index.sum())
# np.ones(len(TDSS_Table)).astype(bool)
# TDSS_Table = TDSS_Table[selection_index]

output_dir = "/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/"
datestr = '2022-04-05'
Vi_panel_dir = output_dir + datestr + "/Vi_plots/"

7997


In [9]:
selected_indexs = np.where(selection_index)[0]

TDSS_Table[selected_indexs]

# where_nonP = selection_index
# selected_indexs = np.where(where_nonP)[0][np.random.randint(0, where_nonP.sum(), 10)]

# TDSS_Table[selected_indexs]

prop_id,ra_GaiaEDR3,dec_GaiaEDR3,plate,mjd,fiber,class,subclass,z,zErr,ELODIE_FILENAME,ELODIE_OBJECT,ELODIE_SPTYPE,ELODIE_BV,ELODIE_TEFF,ELODIE_LOGG,ELODIE_FEH,ELODIE_Z,ELODIE_Z_ERR,ELODIE_Z_MODELERR,ELODIE_RCHI2,ELODIE_DOF,Z_NOQSO,Z_ERR_NOQSO,CLASS_NOQSO,SUBCLASS_NOQSO,class_SDSSDR12,umag,e_umag,gmag,e_gmag,rmag,e_rmag,imag,e_imag,zmag,e_zmag,zsp,e_zsp,f_zsp,zph,e_zph,avg_zph,pmRA,e_pmRA,pmDE,e_pmDE,SpObjID,spType,spCl,subClass_SDSSDR12,ra_epoch2000,dec_epoch2000,ra_error,dec_error,l_GaiaEDR3,b_GaiaEDR3,Z_GaiaEDR3,U_GaiaEDR3,V_GaiaEDR3,W_GaiaEDR3,parallax,parallax_error,parallax_over_error,pm,pmra_GaiaEDR3,pmra_error,pmdec,pmdec_error,astrometric_n_good_obs_al,astrometric_gof_al,astrometric_chi2_al,astrometric_excess_noise,astrometric_excess_noise_sig,astrometric_params_solved,pseudocolour,pseudocolour_error,visibility_periods_used,ruwe,duplicated_source,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_mag,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_mag,phot_rp_mean_flux,phot_rp_mean_mag,phot_bp_rp_excess_factor,bp_rp,dr2_radial_velocity,dr2_radial_velocity_error,dr2_rv_nb_transits,dr2_rv_template_teff,dr2_rv_template_logg,panstarrs1,sdssdr13,skymapper2,urat1,phot_g_mean_mag_error,phot_bp_mean_mag_error,phot_rp_mean_mag_error,phot_g_mean_mag_corrected,phot_g_mean_mag_error_corrected,phot_g_mean_flux_corrected,phot_bp_rp_excess_factor_corrected,ra_epoch2000_error,dec_epoch2000_error,ra_dec_epoch2000_corr,rgeo,b_rgeo_GaiaEDR3,B_rgeo_GaiaEDR3a,rpgeo,b_rpgeo_GaiaEDR3,B_rpgeo_GaiaEDR3a,isDrake,Drake_CRTS,Drake_Vmag,Drake_Per,Drake_Vamp,Drake_Cl,Drake_f_Cl,Drake_Simbad,CSSID,CSS_Nepochs,CSS_mag,CSS_magerr,ZTF_GroupID,ZTF_g_Nepochs,ZTF_g_mag,ZTF_g_magerr,ZTF_r_Nepochs,ZTF_r_mag,ZTF_r_magerr,ZTF_i_Nepochs,ZTF_i_mag,ZTF_i_magerr,CSSLC,ZTFLC,CSS_or_ZTF,CSS_and_ZTF,PyHammerSpecType,PyHammerRV,CaK,CaK_var,Cadel,Cadel_var,CaI4217,CaI4217_var,Gband,Gband_var,Hgam,Hgam_var,FeI4383,FeI4383_var,FeI4404,FeI4404_var,Hbeta,Hbeta_var,MgI,MgI_var,NaD,NaD_var,CaI6162,CaI6162_var,Halpha,Halpha_var,CaH2,CaH2_var,CaH3,CaH3_var,TiO5,TiO5_var,VO7434,VO7434_var,VO7445,VO7445_var,VO-B,VO-B_var,VO7912,VO7912_var,Rb-B,Rb-B_var,NaI,NaI_var,TiO8,TiO8_var,TiO8440,TiO8440_var,Cs-A,Cs-A_var,CaII8498,CaII8498_var,CrH-A,CrH-A_var,CaII8662,CaII8662_var,FeI8689,FeI8689_var,FeH,FeH_var,C2-4382,C2-4382_var,C2-4737,C2-4737_var,C2-5165,C2-5165_var,C2-5636,C2-5636_var,CN-6959,CN-6959_var,CN-7088,CN-7088_var,CN-7259,CN-7259_var,CN-7820,CN-7820_var,CN-8067,CN-8067_var,CN-8270,CN-8270_var,WD-Halpha,WD-Halpha_var,WD-Hbeta,WD-Hbeta_var,WD-Hgamma,WD-Hgamma_var,W_Halpha,W_Halpha_error,ViCompleted,CSS_lc_id,CSS_P,CSS_logProb,CSS_Amp,CSS_t0,CSS_Mt,CSS_a95,CSS_lc_skew,CSS_Chi2,CSS_brtcutoff,CSS_brt10per,CSS_fnt10per,CSS_fntcutoff,CSS_errmn,CSS_ferrmn,CSS_ngood,CSS_nrejects,CSS_nabove,CSS_nbelow,CSS_Tspan100,CSS_Tspan95,CSS_isAlias,CSS_time_whittened,CSS_VarStat,CSS_Con,CSS_m,CSS_b_lin,CSS_chi2_lin,CSS_a,CSS_b_quad,CSS_c,CSS_chi2_quad,ZTF_g_lc_id,ZTF_g_P,ZTF_g_logProb,ZTF_g_Amp,ZTF_g_t0,ZTF_g_Mt,ZTF_g_a95,ZTF_g_lc_skew,ZTF_g_Chi2,ZTF_g_brtcutoff,ZTF_g_brt10per,ZTF_g_fnt10per,ZTF_g_fntcutoff,ZTF_g_errmn,ZTF_g_ferrmn,ZTF_g_ngood,ZTF_g_nrejects,ZTF_g_nabove,ZTF_g_nbelow,ZTF_g_Tspan100,ZTF_g_Tspan95,ZTF_g_isAlias,ZTF_g_time_whittened,ZTF_g_VarStat,ZTF_g_Con,ZTF_g_m,ZTF_g_b_lin,ZTF_g_chi2_lin,ZTF_g_a,ZTF_g_b_quad,ZTF_g_c,ZTF_g_chi2_quad,ZTF_r_lc_id,ZTF_r_P,ZTF_r_logProb,ZTF_r_Amp,ZTF_r_t0,ZTF_r_Mt,ZTF_r_a95,ZTF_r_lc_skew,ZTF_r_Chi2,ZTF_r_brtcutoff,ZTF_r_brt10per,ZTF_r_fnt10per,ZTF_r_fntcutoff,ZTF_r_errmn,ZTF_r_ferrmn,ZTF_r_ngood,ZTF_r_nrejects,ZTF_r_nabove,ZTF_r_nbelow,ZTF_r_Tspan100,ZTF_r_Tspan95,ZTF_r_isAlias,ZTF_r_time_whittened,ZTF_r_VarStat,ZTF_r_Con,ZTF_r_m,ZTF_r_b_lin,ZTF_r_chi2_lin,ZTF_r_a,ZTF_r_b_quad,ZTF_r_c,ZTF_r_chi2_quad,CSS_mean,CSS_median,CSS_std,CSS_weighted_mean,CSS_weighted_std,CSS_skewness,CSS_kurtosis,CSS_shapiro_w,CSS_quartile31,CSS_stetson_j,CSS_stetson_k,CSS_stetson_l,CSS_eta,CSS_cumsum,CSS_half_mag_amplitude_ratio,CSS_alarm,ZTF_g_mean,ZTF_g_median,Z

In [14]:
vi.get_Vi_panels(selected_indexs, TDSS_Table, datestr, copy=True, moveDir='/Users/benjaminroulston/Desktop/TDSS_VarStar_NonP_Vi/')

# vi.get_Vi_panels(selected_indexs, TDSS_Table, datestr, copy=True, moveDir='/Users/benjaminroulston/Desktop/rand10_nonP/')

7997it [01:06, 119.63it/s]


array(['/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-05/Vi_plots/1_000.01471+20.15233_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-05/Vi_plots/8_000.04566+33.45147_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-05/Vi_plots/15_000.09801+24.60530_Vi.pdf',
       ...,
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-05/Vi_plots/21251_359.97082+26.55765_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-05/Vi_plots/21252_359.97641+23.08629_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-05/Vi_plots/21253_359.97856+23.93512_Vi.pdf'],
      dtype='<U133')